# Qwen3-TTS на Colab: смоук-тест MLX/CUDA и замер RTF

Проверяем, поедет ли генерация аудиокниг на бесплатной **Tesla T4** — тем же кодом, что и на маке
(`rab/qwen_tts_stress.py`), но на `mlx[cuda]` вместо Metal.

Что делает ноутбук:

1. ставит `mlx[cuda]==0.32.3` + `mlx-audio==0.5.8` — linux-колёса `manylinux_2_35_x86_64`, без сборки из исходников;
2. проверяет, что MLX реально считает на GPU, а не на CPU;
3. берёт ваш образец голоса (WAV 5–15 с) и его точную расшифровку;
4. синтезирует **одну фразу**, затем четыре, и считает **RTF** — секунды синтеза на секунду звука;
5. выдаёт вердикт: сколько часов книги успеет синтезировать одна сессия Colab (≈11 рабочих часов из 12).

**Перед запуском:** Runtime → Change runtime type → **T4 GPU**.
Тайминг: ~1 мин установка, ~2–4 мин первое скачивание модели (~4,6 ГБ, дальше из кэша), ~1–2 мин на фразу.

> Замер RTF — единственный по-настоящему неизвестный параметр переноса. Всё остальное (зависимости,
> хардкоды путей, установка моделей) уже проверено: колёса под Python 3.12 есть, CUDA-рантайм тянется
> колёсами, синтез-код бэкенд-агностичен.


## Шаг 0. Окружение

In [ ]:
import os

# Кэш свёрток MLX/CUDA: по умолчанию 128, и генерация падает на середине с
# «Cache thrashing is happening, please set the environment variable
#  MLX_CUDA_CONV_CACHE_SIZE to a larger value than 128».
# Это потолок, а не резервация: память берётся по мере надобности, поэтому на T4
# с её ~15 ГБ ставим 8192 МБ. Переменную читает первый импорт mlx — ставим до него.
#  Кэши MLX/CUDA считаются в записях, а не в мегабайтах: по умолчанию свёртки 128, граф 400,
#  внимание 256/64, FFT 128. Когда записей не хватает, MLX печатает «Cache thrashing is happening,
#  please set the environment variable <имя> to a larger value than <N>» и синтез падает на середине
#  книги. Ставим потолки с запасом (это лимит на число записей, а не резервация памяти).
CACHE_ENTRIES = {"MLX_CUDA_CONV_CACHE_SIZE": 12288, "MLX_CUDA_GRAPH_CACHE_SIZE": 8192,
                 "MLX_CUDA_SDPA_CACHE_SIZE": 4096, "MLX_CUDA_SDPA_BACKWARD_CACHE_SIZE": 1024,
                 "MLX_CUDA_FFT_CACHE_SIZE": 1024}
for _name, _value in CACHE_ENTRIES.items():
    os.environ.setdefault(_name, str(_value))

import platform, shutil, subprocess, sys

print("Python:", sys.version.split()[0])
print("glibc :", ".".join(platform.libc_ver()[1].split(".")[:2]) or "?")
print("ffmpeg:", shutil.which("ffmpeg") or "НЕ НАЙДЕН")

gpu, why = "", []
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True, timeout=20)
    gpu = out.stdout.strip()
    if not gpu:
        why.append((out.stderr or "nvidia-smi вернул пусто").strip()[:120])
except Exception as e:
    why.append(str(e))

if not gpu:                     # запасная проверка: CUDA через torch (в образе Colab он есть)
    try:
        import torch
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            gpu = f"{torch.cuda.get_device_name(0)} ({props.total_memory / 2 ** 30:.0f} ГБ, определён через torch)"
    except Exception as e:
        why.append(f"torch: {e}")

print("GPU   :", gpu or "НЕ НАЙДЕН")
if not gpu:
    print("  почему не нашли:", "; ".join(why) or "неизвестно")
    raise AssertionError(
        "GPU не найден. Runtime → Change runtime type → T4 GPU, затем Runtime → Restart session "
        "и Run all заново (смена рантайма перезапускает машину). Без GPU MLX-CUDA не поедет.")
print("  ориентир: T4 = sm_75, ~15 ГБ доступной памяти")
if "T4" not in gpu:
    print("\n⚠️  Похоже, включён не T4: Runtime → Change runtime type → T4 GPU")
    print("    Ноутбук работать не перестанет, но цифры RTF будут другими.")

## Шаг 1. Установка

`mlx[cuda]` под Linux подставляет `mlx-cuda-12` (87 МБ, `manylinux_2_35_x86_64`), который сам тянет
`nvidia-cublas-cu12 12.9.*`, `nvidia-cufft-cu12 11.4.*`, `nvidia-cuda-nvrtc-cu12 12.9.*`, cudnn 9, nccl,
cusolver. Системный CUDA-тулкит Colab не нужен.

Если `import mlx.core` упадёт на `libnvrtc.so.12` / `libcublas.so.12` — раскомментируйте строку ниже.
Пакет `miniaudio` (нужен `mlx-audio` для чтения WAV) может собраться из исходников — это нормально, ~1 мин.


In [ ]:
%pip install -q "mlx[cuda]==0.32.3" "mlx-audio==0.5.8"
# %pip install -q nvidia-cuda-nvrtc-cu12 nvidia-cublas-cu12

## Шаг 2. Проверка: MLX видит GPU

Важно именно `default device: gpu`. Если увидите `cpu` — генерация пойдёт на процессоре и RTF будет
бессмысленным (десятки секунд на фразу).

Про T4 (sm_75): bf16 карта нативно не умеет, а `load_model` в `mlx-audio` не принимает `dtype` —
веса загрузятся как есть. Если в аудио появится треск или шум, это как раз про несовпадение формата;
в таком случае идите в torch-ноутбук (план Б).


In [ ]:
import importlib.metadata as md

import mlx.core as mx
import numpy as np

for pkg in ("mlx", "mlx-cuda-12", "mlx-audio"):
    try:
        print(f"{pkg:14s}", md.version(pkg))
    except Exception:
        print(f"{pkg:14s} НЕ УСТАНОВЛЕН" + ("   ← это Metal-сборка, на Linux GPU не поедет"
                                            if pkg == "mlx-cuda-12" else ""))

device = mx.default_device()
on_gpu = "gpu" in str(device).lower()          # repr выглядит как «Device(gpu, 0)»
print("default device:", device, "| это GPU:", on_gpu)
try:
    print("device info   :", mx.device_info())
except Exception as e:
    print("device info   :", e)

a = mx.random.normal((1024, 1024), dtype=mx.float16)
b = mx.random.normal((1024, 1024), dtype=mx.float16)
s = float(mx.sum(a @ b))
print("fp16 matmul   :", f"{s:.1f}", "ok" if np.isfinite(s) else "NaN — плохой знак")

assert on_gpu, f"MLX выбрал CPU ({device}) — генерация будет бессмысленно медленной"
print("\n✓ GPU на месте")

## Шаг 3. Образец голоса

Нужен WAV 5–15 секунд: один голос, без музыки и шума, моно. Плюс **точная** расшифровка того, что в нём
сказано, — это часть входного формата модели (in-context cloning), и без текста качество заметно падает.

У вас на маке уже лежат готовые образцы вместе с их текстами:

```
~/Downloads/colab-ref/narrator.wav       9,4 с, PCM float32, 24 кГц, моно
~/Downloads/colab-ref/narrator.json      описание голоса + точная расшифровка
~/Downloads/colab-ref/knyaz_myshkin.wav  11,2 с, то же самое
~/Downloads/colab-ref/knyaz_myshkin.json
```

Выберите `narrator.wav` в диалоге ниже.
Если файл уже лежит в `/content` (например, подтянут с Google Drive) — просто задайте
`REF_AUDIO = "/content/narrator.wav"` в следующей ячейке и не вызывайте загрузку.


In [ ]:
from google.colab import files

print("Выберите narrator.wav (WAV 5–15 с, один голос, без музыки)")
uploaded = files.upload()
REF_AUDIO = list(uploaded.keys())[0]
print("Образец:", REF_AUDIO)

In [ ]:
import os, urllib.request

from mlx_audio.utils import load_audio

# Если файл уже на диске — переопределите путь вручную:
# REF_AUDIO = "/content/narrator.wav"
if not os.path.exists(REF_AUDIO):
    REF_AUDIO = "/content/clone.wav"
    if not os.path.exists(REF_AUDIO):
        urllib.request.urlretrieve(
            "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-TTS-Repo/clone.wav", REF_AUDIO)
    print("!! Используется демо-образец из репозитория Qwen3-TTS (английский):")
    print("   RTF померит, но качество русского на нём будет плохим — это аварийный вариант.")

FALLBACK = REF_AUDIO.endswith("clone.wav")
wave = load_audio(REF_AUDIO, sample_rate=24000)          # тот же загрузчик, что и в синтезе
sec = wave.shape[0] / 24000
print(f"{REF_AUDIO}: {sec:.1f} с, {int(wave.shape[0])} сэмплов, пик {float(mx.max(mx.abs(wave))):.3f}")
assert 3 <= sec <= 25, "образец должен быть 5–15 с (допустимо 3–25)"
if float(mx.max(mx.abs(wave))) < 0.01:
    print("⚠️  образец почти тишина — проверьте файл")

## Шаг 4. Точная расшифровка и тестовый текст

**Расшифровка** (REF_TEXT) должна слово в слово совпадать с образцом. Ниже подставлен текст из
`narrator.json` — тот, которым этот образец и был сгенерирован.

**Тестовый текст** — с ударениями в формате «+ перед гласной»: скрипт `rab/qwen_tts_stress.py` сам
переводит `+а` в `а́` (U+0301 после гласной), а над «ё» ударение убирает. Так удобнее: видно,
где именно поставлено ударение, и не надо ловить невидимые комбинирующие символы.

Текста четыре абзаца: первый пойдёт в замер «одной фразы», все четыре — в замер на четырёх.


In [ ]:
NARRATOR_REF_TEXT = 'Усмехнулся тоже и черномазый. Белокурый несколько удивился, что ему удалось сказать довольно, впрочем, плохой каламбур.'

# Текст демо-образца (проверьте на слух; для замера скорости точность не важна):
DEMO_REF_TEXT = ("Okay. Yeah. I resent you. I love you. I respect you. "
                 "But you know what? You blew it! And thanks to you.")

REF_TEXT = DEMO_REF_TEXT if FALLBACK else NARRATOR_REF_TEXT
print("REF_TEXT:", REF_TEXT[:100], "…")

TEST_ONE = 'З+амок сто+ит на холм+е, а м+астер почин+ил зам+ок в двер+и. Князь Лев Никол+аевич М+ышкин верн+улся в Петерб+ург р+анним +утром и ср+азу прошёл в сво+ю кварт+иру на Лит+ейном просп+екте.'

TEST_FOUR = "\n".join([
    'З+амок сто+ит на холм+е, а м+астер почин+ил зам+ок в двер+и. Князь Лев Никол+аевич М+ышкин верн+улся в Петерб+ург р+анним +утром и ср+азу прошёл в сво+ю кварт+иру на Лит+ейном просп+екте.',
    'В нач+але апр+еля сто+яли +ясные мор+озные дни, и Н+евский просп+ект был ещё покр+ыт лёгким сух+им снег+ом. Прох+ожие тороп+ились, подним+ая вор+отники, а изв+озчики бран+ились на х+олод и гололёд.',
    'Он вошёл в небольшу+ю к+омнату с выс+окими +окнами и ув+идел на стол+е развёрнутый журн+ал, очк+и и ост+ывший чай. За +окном крич+али вор+обьи, и +этот прост+ой звук неожид+анно успок+оил ег+о.',
    'Д+ело б+ыло решен+о, и остав+алось т+олько подпис+ать бум+аги. Он д+олго смотр+ел на стр+оки, пот+ом аккур+атно слож+ил лист вч+етверо и убр+ал ег+о во вн+утренний карм+ан сюртук+а.',
])
print(f"\nФраза: {len(TEST_ONE)} символов; четыре абзаца: {len(TEST_FOUR)} символов")

# Самопроверка разметки: «+» обязан стоять вплотную ПЕРЕД гласной.
import re as _re
_VOW = "аеёиоуыэюяАЕЁИОУЫЭЮЯ"
for _name, _t in (("TEST_ONE", TEST_ONE), ("TEST_FOUR", TEST_FOUR)):
    _bad = _re.findall(r"\+(?![" + _VOW + "])", _t)
    assert not _bad, f"{_name}: «+» не перед гласной: {_bad}"
print("разметка ударений корректна: каждый «+» стоит прямо перед гласной")

## Шаг 5. Хелперы (скопированы из `rab/qwen_tts_stress.py` без изменений)

`normalize_stress` — «+» → U+0301; `split_text` — нарезка по абзацам и предложениям не длиннее
`max_chars` (в скрипте по умолчанию 250 символов).


In [ ]:
import re

ACUTE = "́"
VOWELS = "аеёиоуыэюяАЕЁИОУЫЭЮЯ"


def normalize_stress(text: str) -> str:
    """«+» перед гласной → U+0301 после неё."""
    text = re.sub(rf"\+([{VOWELS}])", rf"\1{ACUTE}", text)
    # Ударение над «ё» не нужно.
    return re.sub(rf"([ёЁ]){ACUTE}", r"\1", text)


def split_text(text: str, max_chars: int) -> list:
    """Режет текст на куски по абзацам и предложениям, не длиннее max_chars."""
    chunks = []
    for para in text.splitlines():
        para = " ".join(para.split())
        if not para:
            continue
        cur = ""
        for sent in re.split(r"(?<=[.!?…»])\s+", para):
            while len(sent) > max_chars:      # очень длинное предложение режем по запятой или пробелу
                cut = max(sent.rfind(", ", 0, max_chars), sent.rfind(" ", 0, max_chars))
                cut = cut + 1 if cut > 0 else max_chars
                if cur:
                    chunks.append(cur)
                    cur = ""
                chunks.append(sent[:cut].strip())
                sent = sent[cut:].strip()
            if cur and len(cur) + 1 + len(sent) > max_chars:
                chunks.append(cur)
                cur = sent
            else:
                cur = f"{cur} {sent}".strip()
        if cur:
            chunks.append(cur)
    return chunks


print(normalize_stress("За+мок стои+т на холме+."))
print("кусков:", len(split_text(normalize_stress(TEST_FOUR), 250)))

## Шаг 6. Модель и кодирование образца

`MODEL_FILES` — только то, что нужно для синтеза (без `lora/`, `meter/`, `eval/`, `scripts/` из
репозитория весов). ~4,6 ГБ fp16 → в MLX. Первый запуск скачивает в `~/.cache/huggingface`,
при перезапуске рантайма кэш в Colab теряется — это нормально.


In [ ]:
import time

from mlx_audio.tts.utils import load_model

MODEL = "siriusfreak/qwen3-tts-12hz-1.7b-ru-stress-cf"
MODEL_FILES = [
    "config.json",
    "generation_config.json",
    "preprocessor_config.json",
    "tokenizer_config.json",
    "vocab.json",
    "merges.txt",
    "model.safetensors",
    "speech_tokenizer/*",
]

t0 = time.time()
model = load_model(MODEL, allow_patterns=MODEL_FILES)
print(f"Модель загружена за {time.time() - t0:.0f} с, sample_rate={model.sample_rate}")

t0 = time.time()
ref_wave = load_audio(REF_AUDIO, sample_rate=model.sample_rate)
print(f"Образец закодирован за {time.time() - t0:.1f} с → {tuple(ref_wave.shape)}")

## Шаг 7. Одна фраза → RTF

RTF = секунды синтеза / секунды звука. Меньше 1 — быстрее реального времени.
`model.generate(...)` — генератор `GenerationResult`, у каждого куска поле `.audio` (`mx.array`).
Ровно этот же вызов делает `rab/qwen_tts_stress.py`.


In [ ]:
import json
from pathlib import Path

import numpy as np
from IPython.display import Audio, display
from scipy.io import wavfile

OUT = Path("/content/out")
OUT.mkdir(parents=True, exist_ok=True)
sr = model.sample_rate


def synth(chunk, seed=42):
    """Один кусок текста → (float32-массив, секунды синтеза)."""
    mx.random.seed(seed)
    t = time.time()
    parts = [
        np.asarray(r.audio, dtype=np.float32)
        for r in model.generate(
            text=chunk,
            lang_code="russian",
            ref_audio=ref_wave,
            ref_text=REF_TEXT,
            temperature=0.9,
        )
    ]
    wav = np.concatenate(parts) if parts else np.zeros(0, dtype=np.float32)
    return wav, time.time() - t


def save(path, wav):
    wavfile.write(str(path), sr, (np.clip(wav, -1.0, 1.0) * 32767).astype(np.int16))


phrase = normalize_stress(TEST_ONE)
wav, dt = synth(phrase, seed=43)
sec = len(wav) / sr
print("Фраза:", phrase[:70], "…")
print(f"аудио {sec:.1f} с за {dt:.1f} с → RTF {dt / sec:.2f} ({sec / dt:.1f}× быстрее реального времени)")
save(OUT / "one.wav", wav)
display(Audio(str(OUT / "one.wav")))

## Шаг 8. Четыре фразы → устойчивый RTF и проекция

Одна фраза может попасть в «холодный» прогон, поэтому считаем четыре подряд. RTF здесь взвешен по
длине звука (длинные куски весят больше). Дальше — проекция на книгу: **часы синтеза = часы звука × RTF**.


In [ ]:
chunks = split_text(normalize_stress(TEST_FOUR), 250)
print(f"Кусков: {len(chunks)}\n")

rows = []
pieces = []
run_t0 = time.time()
for i, chunk in enumerate(chunks, 1):
    wav, dt = synth(chunk, seed=42 + i)
    sec = len(wav) / sr
    rows.append({"i": i, "chars": len(chunk), "audio_s": round(sec, 2),
                 "synth_s": round(dt, 2), "rtf": round(dt / sec, 3)})
    print(f"[{i}/{len(chunks)}] {len(chunk):3d} симв. → {sec:5.1f} с звука за {dt:5.1f} с: RTF {dt / sec:.2f}")
    pieces.append(wav)
    pieces.append(np.zeros(int(sr * 0.35), dtype=np.float32))   # пауза между кусками

total_audio = sum(r["audio_s"] for r in rows)
total_synth = sum(r["synth_s"] for r in rows)
rtf_mean = total_synth / total_audio
print(f"\nИтого: {total_audio:.1f} с звука за {total_synth:.1f} с (полный прогон {time.time() - run_t0:.0f} с)")
print(f"RTF (взвешенный)          : {rtf_mean:.2f}")
print(f"Книга 10 ч звука          : ≈ {10 * rtf_mean:.1f} ч синтеза")
print(f"Влезает в сессию 12 ч     : ≈ {11 / rtf_mean:.1f} ч звука")

save(OUT / "four.wav", np.concatenate(pieces))
metrics = {
    "model": MODEL,
    "device": str(mx.default_device()),
    "ref_audio": REF_AUDIO,
    "chunks": rows,
    "rtf_weighted": round(rtf_mean, 3),
    "book_10h_synth_hours": round(10 * rtf_mean, 1),
    "max_audio_hours_per_session": round(11 / rtf_mean, 1),
}
(OUT / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
display(Audio(str(OUT / "four.wav")))
print("метрики:", OUT / "metrics.json")

## Шаг 9. Вердикт

In [ ]:
rtf = metrics["rtf_weighted"]
if rtf < 0.3:
    verdict = f"отлично: книга 10 ч ≈ {10 * rtf:.1f} ч синтеза, можно гнать полный пайплайн в Colab"
elif rtf < 0.7:
    verdict = f"рабочо: книга 10 ч ≈ {10 * rtf:.1f} ч; в сессию 12 ч влезает ≈ {11 / rtf:.1f} ч звука"
elif rtf < 1.0:
    verdict = (f"на грани: книга 10 ч ≈ {10 * rtf:.1f} ч, за сессию только ≈ {11 / rtf:.1f} ч звука — "
               "делить книгу на несколько запусков (resume по progress.json есть)")
else:
    verdict = (f"плохо: {rtf:.2f}× медленнее реального времени, за 12 ч сессии только ≈ {11 / rtf:.1f} ч звука. "
               "T4 не годится: нужен A100/L4 (Colab Pro) или план Б — torch")
print(f"RTF = {rtf:.2f}\nВЕРДИКТ: {verdict}")

## Шаг 10. Что дальше — полный пайплайн в Colab

Смоук-тест отвечает только за скорость. Если RTF устроил, дальше всё уже известно: пять правок,
которые превращают маковый репозиторий в колабовский.

```bash
!git clone https://github.com/davnozdu/qwen-epub-audiobook.git
%cd qwen-epub-audiobook

# 1) requirements: mlx-metal на Linux не ставится вообще → mlx[cuda]
!sed -i 's#mlx-metal==0.32.3#mlx[cuda]==0.32.3#' requirements.txt requirements-mlx.txt

# 2) пять хардкодов ffmpeg/ffprobe под Homebrew → системный
!sed -i 's#/opt/homebrew/bin/#/usr/bin/#g' \
    audiobook_epub.py normalize_book_audio.py mix_book_music.py synthesize_qwen_mlx.py
!grep -rn "/opt/homebrew" *.py || echo "хардкодов не осталось"

# 3) setup_audiobook.py НЕ запускать (это установщик для macOS: venv, brew, guard на Apple Silicon);
#    модели качает сам HF-кэш из design_model_files.json / stress_model_files.json

# 4) ключ Ollama нужен ТОЛЬКО стадиям LLM-разметки: prepare_qwen.py run и prepare_speech.py run
#    (getpass на пустом stdin в Colab даёт EOFError). Всё остальное ключа не требует:
#    synthesize_qwen_mlx.py — вообще нигде; prepare_silero.py — при --llm-disputes off (по умолчанию)
#    и даже при ask он ловит EOFError и идёт без LLM; prepare_speech.py offline — без LLM.
#    Нужен ключ — только в окружение:
import os; os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")   # или свой ключ
#    Не нужен — проходить эти стадии командами advance (пишет request.json, запрос исполняется где угодно)
#    и offline; плюс --cast cast.json пропускает первую стадию определения ролей.

# 5) START.command / TEST-7MIN.command не использовать — они зовут /opt/homebrew/opt/python@3.11
```

Порядок запуска:

```bash
python -c "import mlx.core as mx; print(mx.default_device(), mx.device_info())"   # ждём gpu
python prepare_silero.py      # ударения (torch в Colab уже есть)
python prepare_qwen.py        # манифест озвучки по главам
python synthesize_qwen_mlx.py <манифест>   # синтез; resume по progress.json и sha256 сегментов
python normalize_book_audio.py / mix_book_music.py   # сведение и громкость
```

Практика:

- выход держать на **Google Drive** (`/content/drive/MyDrive/...`): 12 ч сессии не хватит на книгу целиком,
  а `progress.json` + sha256 позволяют продолжить в следующем запуске, не пересинтезируя готовое;
- модель скачивать один раз за сессию (~4,6 ГБ, 2–4 мин) — расписание «одна книга = одна сессия» самое дешёвое;
- ffmpeg 4.4.2 в Colab уже есть в `/usr/bin/ffmpeg`, отдельно ставить не нужно.

## Tips

- **Точность расшифровки** решает: текст образца слово в слово, иначе клонирование заметно хуже.
- **Качество образца**: чистая речь, один голос, 5–15 с. Ваши `narrator.wav` (9,4 с) и `knyaz_myshkin.wav`
  (11,2 с) — ровно тот формат, что нужен (PCM float32, 24 кГц, моно).
- **Ударения**: «+» перед гласной или U+0301 после неё; над «ё» не нужно. Без разметки модель ставит
  ударения сама, но на редких словах ошибается — для аудиокниги это слышно.
- **Скачать результат**: `from google.colab import files; files.download("/content/out/four.wav")`.
- Если цифры RTF не устроят — план Б в `qwen_tts_colab_smoke_torch.ipynb` (тот же голос и текст через
  torch/CUDA и официальный `qwen-tts`), сравнение двух бэкендов на одной карте даёт прямой ответ,
  какой из них брать в продакшн.

---

Замер на T4 (sm_75, ~15 ГБ доступной памяти): модель 4,6 ГБ + CUDA-рантайм помещаются с большим запасом.
